# 01 - Data Exploration

First look at the raw material of this project: the **PaySim** synthetic
mobile-money dataset that simulates M-Pesa transactions.

**Goal:** understand the schema, verify data quality (missing values,
duplicates, impossible balances) and get a first feel for the class imbalance.

| Column | Meaning |
|---|---|
| `step` | 1 step = 1 simulated hour (1-744 = 31 simulated days) |
| `type` | PAYMENT, TRANSFER, CASH_OUT, CASH_IN, DEBIT |
| `amount` | transaction amount |
| `nameOrig` / `nameDest` | origin / destination account |
| `oldbalanceOrg` / `newbalanceOrig` | origin balance before / after |
| `oldbalanceDest` / `newbalanceDest` | destination balance before / after |
| `isFraud` | **target** - 1 for a fraudulent transaction |
| `isFlaggedFraud` | simulator flag for "odd" large transfers (rarely set) |

> **Prerequisite:** run `python main.py` once so
> `data/processed/cleaned_transactions.csv` exists. Notebooks read the
> *cleaned* copy (lower-cased, validated), not the raw Kaggle download.

Run order: `01 -> 02 -> 03 -> 04 -> 05`.

In [ ]:
import sys
from pathlib import Path

# Make 'import src...' work no matter where the kernel was started from.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
FEATURED_PATH = PROJECT_ROOT / "data" / "processed" / "featured_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"

print("Project root:", PROJECT_ROOT)

## 1. Load a working sample

The cleaned file holds **6.36M rows (~499 MB)**. To keep this notebook
interactive we read a 200k-row sample; set `RANDOM_SAMPLE = True` to draw a
uniform random sample over the whole file instead.

In [ ]:
SAMPLE_ROWS = 200_000    # rows to keep for this notebook
RANDOM_SAMPLE = False    # True = uniform random sample (streams the whole file, slower)

if RANDOM_SAMPLE:
    # Streams the full cleaned CSV one chunk at a time and keeps a random
    # slice: memory-bounded and unbiased, but slower than the head sample.
    from src.detection.fraud_model import sample_processed_data
    df = sample_processed_data(max_rows=SAMPLE_ROWS, random_state=42)
else:
    # The cleaned file is ordered by simulation time, so the head covers the
    # first hours of the simulated month. Fast; plenty for structural EDA.
    df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)

print(df.shape, "| fraud rows in sample:", int(df["isfraud"].sum()))

In [ ]:
df.head()

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "first_value": [df[c].iloc[0] for c in df.columns],
})
profile

## 2. Data-quality checks

The cleaning stage (`src/data/data_cleaning.py`) lower-cases the headers,
drops duplicates, imputes missing values and rejects impossible rows. Here we
confirm what the cleaned file actually contains.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if len(missing) else "  none")

print("\nDuplicate rows:", int(df.duplicated().sum()))

negatives = (df.select_dtypes("number") < 0).sum()
negatives = negatives[negatives > 0]
print("\nNegative numeric values (should be none):")
print("  none" if not len(negatives) else negatives)

In [ ]:
# Target balance: PaySim is heavily imbalanced towards legitimate traffic.
counts = df["isfraud"].value_counts().rename({0: "legitimate", 1: "fraud"})
share = (df["isfraud"].value_counts(normalize=True) * 100).round(4)
pd.DataFrame({"transactions": counts, "share_%": share})

In [ ]:
df["type"].value_counts().to_frame("transactions")

## 3. Amounts

Transaction amounts are heavily right-skewed, so percentiles are far more
informative than a mean.

In [ ]:
df["amount"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame().T

In [ ]:
by_type = df.groupby("type")["isfraud"].agg(transactions="count", fraud="sum")
by_type["fraud_rate_%"] = (100 * by_type["fraud"] / by_type["transactions"]).round(4)
by_type.sort_values("fraud_rate_%", ascending=False)

## 4. Time coverage

`step` is an hourly clock, so it can be folded into hour-of-day and day-of-month
features (see `03_feature_engineering.ipynb`).

In [ ]:
print(f"step range: {int(df['step'].min())} .. {int(df['step'].max())}")
span_hours = int(df["step"].max() - df["step"].min() + 1)
print(f"= {span_hours / 24:.1f} simulated days")

per_day = df.assign(day=(df["step"] - 1) // 24).groupby("day").size()
ax = per_day.plot(figsize=(10, 3), title="Transactions per simulated day")
ax.set_xlabel("simulated day")
ax.set_ylabel("transactions")

In [ ]:
balance_cols = ["oldbalanceorg", "newbalanceorig", "oldbalancedest", "newbalancedest"]
df[balance_cols].describe().T

## Summary

- **Schema:** one row = one transaction, 11 columns; the cleaned file has no
  missing values, no duplicates and no impossible (negative) amounts, so
  cleaning is mostly renaming/validation rather than repair.
- **Severe class imbalance:** fraud is a tiny fraction of all transactions.
  Accuracy will be meaningless - precision, recall and ROC-AUC are the metrics
  that matter (developed in `05_model_evaluation.ipynb`).
- **Fraud only ever occurs for money-leaving types** - quantified in
  `02_fraud_eda.ipynb`.
- **Time features are available** from `step` (hour of day, day, velocity).